In [ ]:
import numpy as np
import base64
import struct
from cobs import cobs
from collections import namedtuple
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.pyplot import plot
from tqdm.notebook import tqdm
import scipy.signal
from scipy.io import savemat
# import imageio
import pandas as pd
import scipy.io as spio
import os

from totalsync_utils import decode_b64_files

In [ ]:
# Format package
DataPacketDesc = {'type': 'B',
				  'size': 'B',
				  'crc16': 'H',
				  'packetID': 'I',
				  'us_start': 'I',
				  'us_end': 'I',
				  'analog': '8H',
				  'states': '8l',
				  'digitalIn': '2H',
				  'digitalOut': '3B',
				  'padding': 'x'}


DataPacket = namedtuple('DataPacket', DataPacketDesc.keys())
DataPacketStruct = '<' + ''.join(DataPacketDesc.values())
DataPacketSize = struct.calcsize(DataPacketStruct)

# package with non-digital data
dtype_no_digital = [
	('type', np.uint8),
	('size', np.uint8),
	('crc16', np.uint16),
	('packetID', np.uint32),
	('us_start', np.uint32),
	('us_end', np.uint32),
	('analog', np.uint16, (8, )),
	('states', np.uint32, (8, ))]

# DigitalIn and DigitalOut
dtype_w_digital = dtype_no_digital + [('digital_in', np.uint16, (16, )), ('digital_out', np.uint16, (16, ))]

# Creating array with all the data (differentiation digital/non digital)
np_DataPacketType_noDigital = np.dtype(dtype_no_digital)
np_DataPacketType_withDigital = np.dtype(dtype_w_digital)

In [13]:
# function to count the packet number
def count_lines(fp):
	def _make_gen(reader):
		b = reader(2**17)
		while b:
			yield b
			b = reader(2**17)
	with open(fp, 'rb') as f:
		count = sum(buf.count(b'\n') for buf in _make_gen(f.raw.read))
	return count

def unpack_data_packet(dp):
	s = struct.unpack(DataPacketStruct, dp)
	up = DataPacket(type=s[0], size=s[1], crc16=s[2], packetID=s[3], us_start=s[4], us_end=s[5],
						analog=s[6:14], states=s[14:22], digitalIn=s[22], digitalOut=s[23], padding=None)

	return up

## 16/16

In [14]:
Path_non_decoded_files = "/Users/fpbattaglia/Dropbox/Data/Totalsync/477116"
Path_decoded = os.path.join(Path_non_decoded_files, 'decoded')

for filename in os.listdir(Path_non_decoded_files):
    if filename.endswith(".b64"):
        bp=os.path.join(Path_non_decoded_files, filename)

        num_lines = count_lines(bp)
        log_duration = num_lines/1000/60
        print(bp)
        print(f'{num_lines} packets, ~{log_duration:0.2f} minutes')


        # Decode and create new dataset
        data = np.zeros(num_lines, dtype=np_DataPacketType_withDigital)
        print(len(data))
        non_digital_names = list(np_DataPacketType_noDigital.names)

        with open(bp, 'rb') as bf:
            for nline, line in enumerate(tqdm(bf, total=num_lines)):
                bl = cobs.decode(base64.b64decode(line[:-1])[:-1])
                dp = unpack_data_packet(bl)
                data[non_digital_names][nline] = np.frombuffer(bl[:-8], dtype=np_DataPacketType_noDigital)
                digital_arr = np.frombuffer(bl[-8:], dtype=np.uint8)
                data[nline]['digital_in'] = np.hstack([np.unpackbits(digital_arr[1]), np.unpackbits(digital_arr[0])])
                data[nline]['digital_out'] = np.hstack([np.unpackbits(digital_arr[3]), np.unpackbits(digital_arr[2])])

        #Check for packetID jumps
        jumps = np.unique(np.diff(data['packetID']))

       # assert(len(jumps) and jumps[0] == 1)
        data['digital_in']=np.flip(data['digital_in'], 1)
        data['digital_out']=np.flip(data['digital_out'], 1)
        decoded = {"analog":data['analog'], "digitalIn":data['digital_in'], "digitalOut":data['digital_out'], "startTS":data['us_start'], "transmitTS":data['us_end'], "longVar":data['states'], "packetNums":data['packetID']}
        name=os.path.splitext(filename)[0]
        print(name)
        path=os.path.join(Path_decoded, name +'_decoded'+'.mat')
        savemat(path, decoded)

/Users/fpbattaglia/Dropbox/Data/Totalsync/477116/20251118-155459_534.b64
624348 packets, ~10.41 minutes
624348


  0%|          | 0/624348 [00:00<?, ?it/s]

20251118-155459_534
/Users/fpbattaglia/Dropbox/Data/Totalsync/477116/20251120-122334_670.b64
2139534 packets, ~35.66 minutes
2139534


  0%|          | 0/2139534 [00:00<?, ?it/s]

20251120-122334_670
/Users/fpbattaglia/Dropbox/Data/Totalsync/477116/20251118-155315_863.b64
83025 packets, ~1.38 minutes
83025


...

20251118-155315_863
/Users/fpbattaglia/Dropbox/Data/Totalsync/477116/20251118-152602_925.b64
1383156 packets, ~23.05 minutes
1383156


  0%|          | 0/1383156 [00:00<?, ?it/s]

20251118-152602_925
/Users/fpbattaglia/Dropbox/Data/Totalsync/477116/20251118-160736_363.b64
789669 packets, ~13.16 minutes
789669


  0%|          | 0/789669 [00:00<?, ?it/s]

20251118-160736_363


In [15]:
data[0]

np.void((0, 72, 49682, 20510, 24511993, 24512174, [1793,    0, 1653, 2206, 2395, 2216, 2403, 4054], [  0,   0,   0,   0,   0,   0,   0, 182], [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]), dtype=[('type', 'u1'), ('size', 'u1'), ('crc16', '<u2'), ('packetID', '<u4'), ('us_start', '<u4'), ('us_end', '<u4'), ('analog', '<u2', (8,)), ('states', '<u4', (8,)), ('digital_in', '<u2', (16,)), ('digital_out', '<u2', (16,))])

In [10]:
digital_in = data['digital_in']

In [16]:
results = decode_b64_files(Path_non_decoded_files)


Processing 20251118-155459_534.b64...
/Users/fpbattaglia/Dropbox/Data/Totalsync/477116/20251118-155459_534.b64
624348 packets, ~10.41 minutes


100%|██████████| 624348/624348 [00:07<00:00, 87584.18it/s]



Processing 20251120-122334_670.b64...
/Users/fpbattaglia/Dropbox/Data/Totalsync/477116/20251120-122334_670.b64
2139534 packets, ~35.66 minutes


100%|██████████| 2139534/2139534 [00:24<00:00, 87426.00it/s]



Processing 20251118-155315_863.b64...
/Users/fpbattaglia/Dropbox/Data/Totalsync/477116/20251118-155315_863.b64
83025 packets, ~1.38 minutes


100%|██████████| 83025/83025 [00:00<00:00, 86642.38it/s]



Processing 20251118-152602_925.b64...
/Users/fpbattaglia/Dropbox/Data/Totalsync/477116/20251118-152602_925.b64
1383156 packets, ~23.05 minutes


100%|██████████| 1383156/1383156 [00:15<00:00, 86801.44it/s]



Processing 20251118-160736_363.b64...
/Users/fpbattaglia/Dropbox/Data/Totalsync/477116/20251118-160736_363.b64
789669 packets, ~13.16 minutes


100%|██████████| 789669/789669 [00:09<00:00, 86369.28it/s]


In [18]:
results.keys()

dict_keys(['20251118-155459_534', '20251120-122334_670', '20251118-155315_863', '20251118-152602_925', '20251118-160736_363'])

In [19]:
data_package = results['20251118-160736_363']

In [23]:
data['type']

array([0, 0, 0, ..., 0, 0, 0], shape=(789669,), dtype=uint8)

In [22]:
data_package

{'analog': array([[1793,    0, 1653, ..., 2216, 2403, 4054],
        [1793,    0, 1648, ..., 2211, 2403, 4052],
        [1788,    0, 1649, ..., 2211, 2404, 4050],
        ...,
        [1786,    0, 1650, ..., 2204, 2401, 4052],
        [1785,    0, 1644, ..., 2205, 2401, 4051],
        [1785,    0, 1575, ..., 2203, 2401, 4052]],
       shape=(789669, 8), dtype=uint16),
 'digitalIn': array([[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]], shape=(789669, 16), dtype=uint16),
 'digitalOut': array([[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 1, 0],
        [0, 0, 0, ..., 0, 1, 0],
        [0, 0, 0, ..., 0, 1, 0]], shape=(789669, 16), dtype=uint16),
 'startTS': array([ 24511993,  24512993,  24513993, ..., 459942353, 459943353,
        459944353], shape=(78

In [25]:
data_package.keys()

dict_keys(['analog', 'digitalIn', 'digitalOut', 'startTS', 'transmitTS', 'longVar', 'packetNums'])

In [26]:
data.dtype

dtype([('type', 'u1'), ('size', 'u1'), ('crc16', '<u2'), ('packetID', '<u4'), ('us_start', '<u4'), ('us_end', '<u4'), ('analog', '<u2', (8,)), ('states', '<u4', (8,)), ('digital_in', '<u2', (16,)), ('digital_out', '<u2', (16,))])

In [28]:
pin_sheet_file = '/Users/fpbattaglia/src/ofl_2p_analysis/docs/pinSheet.json'

results_pins = decode_b64_files(Path_non_decoded_files, pin_json_path=pin_sheet_file)


Processing 20251118-155459_534.b64...
/Users/fpbattaglia/Dropbox/Data/Totalsync/477116/20251118-155459_534.b64
624348 packets, ~10.41 minutes


100%|██████████| 624348/624348 [00:07<00:00, 86865.12it/s]



Processing 20251120-122334_670.b64...
/Users/fpbattaglia/Dropbox/Data/Totalsync/477116/20251120-122334_670.b64
2139534 packets, ~35.66 minutes


100%|██████████| 2139534/2139534 [00:24<00:00, 86257.12it/s]



Processing 20251118-155315_863.b64...
/Users/fpbattaglia/Dropbox/Data/Totalsync/477116/20251118-155315_863.b64
83025 packets, ~1.38 minutes


100%|██████████| 83025/83025 [00:00<00:00, 86353.21it/s]



Processing 20251118-152602_925.b64...
/Users/fpbattaglia/Dropbox/Data/Totalsync/477116/20251118-152602_925.b64
1383156 packets, ~23.05 minutes


100%|██████████| 1383156/1383156 [00:15<00:00, 86835.49it/s]



Processing 20251118-160736_363.b64...
/Users/fpbattaglia/Dropbox/Data/Totalsync/477116/20251118-160736_363.b64
789669 packets, ~13.16 minutes


100%|██████████| 789669/789669 [00:09<00:00, 86993.34it/s]


In [29]:
data_pins = results_pins['20251118-160736_363']

In [30]:
data_pins.keys()

dict_keys(['startTS', 'transmitTS', 'longVar', 'packetNums', 'GND', 'Wheel Encoder A', 'Wheel Encoder B', 'Wheel Encoder X(check)', 'Wheel Encoder Z(check)', 'Scanner Frame Clock (Input)', 'Finish Trial', 'Reward Tile', 'City Environment', 'Nature Environment', 'Halloween Environment', 'Tunnel 1', 'Tunnel 2', 'New Trial', 'Valve Toggle', 'Barcode (Scanner)', 'OFL Experiment', 'Shock Grid', 'Pre-Shock Experiment', 'Test Shock', 'Ephys Trigger', 'Pin Sync LED', 'Trigger C', 'Lick Detection', 'Breathing Sensor'])